### 데이터 불러오기

In [1]:
import pandas as pd

# 1.데이터 불러오기
X = pd.read_csv(r'C:\miniprj\실습데이터\전처리결과\김해_전처리데이터셋.csv', encoding="cp949")
y = pd.read_csv(r'C:\miniprj\실습데이터\전처리결과\김해_충전량현황종합_전처리.csv', encoding="cp949")

In [2]:
X

,gid,건물1,건물2,건물3,관공서,주차장,경제인구,전기차,충전소,차량속도,lon,lat
0,마라09ab99bb,0,0,0,0,NaN,NaN,0,0,NaN,128.702814,35.288404
1,마라09ba98ba,0,0,0,0,NaN,NaN,0,0,NaN,128.705396,35.277108
2,마라09ba98bb,0,0,0,0,NaN,NaN,0,0,NaN,128.705430,35.279362
3,마라09ba99aa,0,0,0,0,NaN,NaN,0,0,NaN,128.705463,35.281615
4,마라09ba99ab,0,0,0,0,NaN,NaN,0,0,NaN,128.705496,35.283869
...,...,...,...,...,...,...,...,...,...,...,...,...
7673,마마36aa00aa,0,0,0,0,NaN,NaN,0,0,NaN,128.996949,35.287375
7674,마마36aa00ab,0,0,0,0,NaN,NaN,0,0,NaN,128.996991,35.289629
7675,마마36ab00aa,0,0,0,0,NaN,NaN,0,0,NaN,128.999698,35.287341
7676,마마36ab00ab,0,0,0,0,NaN,NaN,0,0,NaN,128.999739,35.289595


In [3]:
y

,충전소명,마지막 충전시작일시,마지막 충전종료일시,총충전량,총이용수,일평균충전량,일평균이용수,LON,LAT,gid
0,장유(서부산) 휴게소,2023/10/30,2023/10/30,16259.41,1036,59.558278,3.794872,128.793845,35.212465,마라17bb91ba
1,장유(서부산) 휴게소,2023/10/30,2023/10/30,27368.69,1839,100.251612,6.736264,128.793845,35.212465,마라17bb91ba
2,롯데마트 장유점,2023/10/29,2023/10/29,17044.62,956,62.434505,3.501832,128.801298,35.193038,마라18ab89ab
3,롯데마트 김해점,2023/10/30,2023/10/30,7466.94,636,27.351429,2.329670,128.882049,35.225546,마라25bb93aa
4,김해금관가야(기장) 휴게소,2023/10/30,2023/10/30,37079.74,2543,135.823223,9.315018,129.002795,35.268229,마라36ba98aa
...,...,...,...,...,...,...,...,...,...,...
62,유하공원공영주차장,2023/10/29,2023/10/29,4.75,2,0.017399,0.007326,128.809137,35.205657,마라19aa90bb
63,지내동공단 공영주차장,2023/10/30,2023/10/30,8.52,5,0.031209,0.018315,128.922396,35.226382,마라29ab93ab
64,지내동공단 공영주차장,2023/10/30,2023/10/30,9.00,5,0.032967,0.018315,128.922396,35.226382,마라29ab93ab
65,김해문화의전당,2023/10/30,2023/10/30,5.68,3,0.020806,0.010989,128.869900,35.243656,마라24ba95aa


### 김해지역 입지 총점 계산

In [4]:
from sklearn.preprocessing import MinMaxScaler

# 2.인덱스 설정
X.set_index('gid', inplace=True)
y.set_index('gid', inplace=True)

selected_cols = ['건물1','건물2','건물3','관공서','주차장','경제인구','전기차','차량속도']

In [5]:
# 3.결측치 처리 (0으로 대체)
X_selected = X[selected_cols].fillna(0)

# 4.MinMax 정규화
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_selected), index=X.index, columns=selected_cols)

target_col = '일평균이용수'  # '일평균충전량' 혹은 '일평균이용수'
df = X_scaled.join(y[target_col], how='inner')

In [6]:
# 5. 상관계수 계산
corr = df.corr()[target_col].drop(target_col)
print('가중치 (상관계수):')
print(corr)

가중치 (상관계수):
건물1    -0.091629
건물2    -0.091204
건물3     0.003840
관공서     0.092136
주차장    -0.118954
경제인구   -0.131525
전기차     0.099947
차량속도   -0.088036
Name: 일평균이용수, dtype: float64


In [7]:
# 6.회귀분석에 사용할 변수 선택
vars = ['건물1','건물2','관공서','주차장','경제인구','전기차','차량속도']
weights = corr[vars].copy()

In [8]:
# 7.총점 계산
X_sub = X_scaled[weights.index]
X_scaled['입지총점'] = X_sub.dot(weights)

# 8.결과 확인
print(X_scaled[['입지총점']].sort_values('입지총점', ascending=False).head(5))

                입지총점
gid                 
마라27ab96aa  0.075926
마라21ab86ab  0.065930
마라23bb96ba  0.063973
마라19aa86ba  0.054027
마라17aa87aa  0.052304


## 랜덤포레스트를 활용한 기존충전소 위치를 제외 신규후보지 예측

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# 9. 회귀분석 준비
X_reg = X_scaled.loc[y.index, weights.index]
y_reg = y[target_col]

# 10. Random Forest 회귀모델 생성 및 학습
model = RandomForestRegressor(
    n_estimators=100,     # 트리 개수
    max_depth=5,          # 트리 최대 깊이 (튜닝 가능)
    random_state=42
)
model.fit(X_reg, y_reg)

# 11. 예측 및 R² 확인
y_pred = model.predict(X_reg)
r2 = r2_score(y_reg, y_pred)
print(f"R² (랜덤포레스트): {r2:.4f}")

# 12. 신규 후보지 예측
new_candidates = X_scaled.loc[~X_scaled.index.isin(y.index)].copy()

if not new_candidates.empty:
    new_candidates_pred = model.predict(new_candidates[weights.index])
    new_candidates['예측_'+target_col] = new_candidates_pred
    new_candidates['총점순위'] = new_candidates['입지총점'].rank(method='dense', ascending=False)
    print("\n신규 후보지 예측 결과 상위 20개:")
    print(new_candidates[['입지총점', '예측_'+target_col, '총점순위']].sort_values('입지총점', ascending=False).head(20))
else:
    print("\n신규 후보지 데이터가 없습니다.")

# 13. 결과 저장
new_candidates.to_csv(r'C:\miniprj\실습데이터\전처리결과\김해_결과총점(회귀분석).csv', index=True, sep=',', encoding='cp949')

R² (랜덤포레스트): 0.6834

신규 후보지 예측 결과 상위 20개:
                입지총점  예측_일평균이용수  총점순위
gid                                  
마라27ab96aa  0.075926   5.474382   1.0
마라21ab86ab  0.065930   4.338138   2.0
마라23bb96ba  0.063973   6.777213   3.0
마라19aa86ba  0.054027   4.193297   4.0
마라17aa87aa  0.052304   4.125779   5.0
마라24bb98aa  0.049884   6.959257   6.0
마라21aa94aa  0.040277   3.370147   7.0
마라24aa98aa  0.037529   3.357629   8.0
마라20ba88bb  0.033402   4.125779   9.0
마라34ba95aa  0.027837   7.951054  10.0
마라24bb97ba  0.026261   3.289859  11.0
마라24ab97aa  0.025250   1.792653  12.0
마라27ba97ab  0.021750   4.125779  13.0
마라22ba91bb  0.020108   4.193297  14.0
마라22ab94ba  0.019161   3.388484  15.0
마마13ab01bb  0.017866   3.388484  16.0
마라14aa94ab  0.017209   2.794219  17.0
마라28aa96ab  0.016522   2.690359  18.0
마라16aa88ba  0.015933   7.516463  19.0
마라24ba91bb  0.015482   3.358836  20.0


In [13]:
new_candidates

,건물1,건물2,건물3,관공서,주차장,경제인구,전기차,차량속도,입지총점,예측_일평균이용수,총점순위
gid,,,,,,,,,,,
마라09ab99bb,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.113076,62.0
마라09ba98ba,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.113076,62.0
마라09ba98bb,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.113076,62.0
마라09ba99aa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.113076,62.0
마라09ba99ab,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.113076,62.0
...,...,...,...,...,...,...,...,...,...,...,...
마마36aa00aa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.113076,62.0
마마36aa00ab,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.113076,62.0
마마36ab00aa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.113076,62.0


### folium 활용 입지총점 상위위치 시각화

In [14]:
import folium

#위경도 데이터가져기기
folium_data = new_candidates.join(X[['lat', 'lon']], how='left')
top_sites = folium_data[folium_data['총점순위'] <= 10.0] #총점순위 10위까지 필터링

# 지도 초기화 (중심은 상위 5개 평균 위치로)
map_center = [top_sites['lat'].mean(), top_sites['lon'].mean()]
m = folium.Map(location=map_center, zoom_start=14)

# 마커 추가
for idx, row in top_sites.iterrows():
    folium.Marker(
        location=(row['lat'], row['lon']),
        popup=f"gid: {idx}<br>입지총점: {row['입지총점']:.2f}",
        icon=folium.Icon(color='red')
    ).add_to(m)
    
#저장
m.save("입지총점_지도.html")    

In [15]:
m